# Analise de Dados - Poke API

Perguntas

- Pokémons com múltiplos tipos e força acima da média

- Abilities exclusivas de pokémons com múltiplos tipos

- Pokémons mais versáteis


## Imports

In [0]:
from pyspark.sql import functions as F


## Iniciar tabelas

In [0]:
df_pokemon = spark.table("workspace.default.pokemon")
df_pokemon_estatisticas = spark.table("workspace.default.pokemon_stats")
df_pokemon_tipos = spark.table("workspace.default.pokemon_type")
df_pokemon_habilidades = spark.table("workspace.default.pokemon_ability")

## Pokémons com múltiplos tipos e força acima da média

In [0]:
# Cria dataframe com a força total de cada pokemons
df_pokemon_forca = df_pokemon_estatisticas.groupBy("pokemon_id").agg(F.sum("base_stat").alias("total_force"))

# filtra as condições
media_forca = df_pokemon_forca.agg(F.avg("total_force")).first()[0]

df_pokemon_forca_acima_media = df_pokemon_forca.filter(F.col("total_force") > media_forca)    

df_pokemons_multi_tipo = df_pokemon_tipos.groupBy("pokemon_id") \
                                .agg(F.countDistinct("type_name").alias("num_types"))\
                                .filter(F.col("num_types") > 1)

# realiza o inner join depois de filtrar os dados 
resultado = df_pokemon_forca_acima_media.join(df_pokemons_multi_tipo, "pokemon_id", "inner").count()

print(f"{resultado} pokemons possuem mais de um tipo e força maior que a média") 

510 pokemons possuem mais de um tipo e força maior que a média


### visualização individual dos dataframes

In [0]:
display(media_forca)

452.1044444444444

In [0]:
display(df_pokemon_forca_acima_media.join(df_pokemons_multiplos_tipos, "pokemon_id", "inner"))

pokemon_id,total_force,num_types
849,502.0,2
10021,600.0,2
10076,700.0,2
208,510.0,2
248,600.0,2
890,690.0,2
171,460.0,2
998,600.0,2
10265,670.0,2
781,517.0,2


In [0]:
display(df_pokemon_forca_acima_media)

pokemon_id,total_force
3,525.0
6,534.0
9,530.0
18,479.0
26,485.0
31,505.0
34,505.0
36,483.0
38,505.0
42,455.0


In [0]:
display(df_pokemons_multi_tipo)

pokemon_id,num_types
1,2
2,2
3,2
6,2
12,2
13,2
14,2
15,2
16,2
17,2


## Abilities exclusivas de pokémons com múltiplos tipos

In [0]:
# IDs dos pokémons com múltiplos tipos
df_pokemons_multi_tipo = df_pokemons_multi_tipo.select("pokemon_id")

# IDs dos pokémons com um único tipo
df_pokemons_unicos_tipos = df_pokemon_tipos.groupBy("pokemon_id") \
    .agg(F.countDistinct("type_name").alias("n_types")) \
    .filter(col("n_types") == 1) \
    .select("pokemon_id")

# Habilidades de pokemons múltiplos tipos
df_hab_multi_tipos_distinct = df_pokemon_habilidades.join(df_pokemons_multi_tipo, "pokemon_id") \
    .select("ability_name").distinct()
    
# Habilidades de pokemons tipos unicos
df_hab_unicos_tipos_distinct = df_pokemon_habilidades.join(df_pokemons_unicos_tipos, "pokemon_id") \
    .select("ability_name").distinct()

# habilidades exclusivas para pokemons de multiplos tipos
df_hab_ex_multi_tipos = df_hab_multi_tipos_distinct.join(df_hab_unicos_tipos_distinct, "ability_name", "left_anti")

display(df_hab_ex_multi_tipos)

ability_name
air-lock
dancer
merciless
corrosion
punk-rock
protosynthesis
hadron-engine
hospitality
surge-surfer
aura-break


## Pokémons mais versáteis

In [0]:
df_pokemon_num_tipo = df_pokemon_tipos.groupBy("pokemon_id").agg(F.countDistinct("type_name").alias("n_types"))
df_pokemon_num_hab = df_pokemon_habilidades.groupBy("pokemon_id").agg(F.countDistinct("ability_name").alias("n_abilities"))

# Join para obter n_types, n_abilities e total_force
df_versatilidade = df_pokemon.select("pokemon_id", "name") \
    .join(df_pokemon_num_tipo, "pokemon_id") \
    .join(df_pokemon_num_hab, "pokemon_id") \
    .join(df_pokemon_forca.select("pokemon_id", "total_force"), "pokemon_id")\
    .withColumn("versatility_score",(F.col("n_types") * 2) + F.col("n_abilities") + (F.col("total_force") / 100)
)

# mostra os 4 pokemons com maior versatilidade
display(df_versatilidade.select("name", "versatility_score").orderBy(col("versatility_score").desc()).limit(5))

name,versatility_score
eternatus-eternamax,16.25
dragapult,13.0
kommo-o-totem,13.0
kommo-o,13.0
archaludon,13.0
